# Fine-tune the small claim verifier (LoRA, free T4)

Trains the BioResearch Agent's claim verifier: **Qwen/Qwen3-1.7B** (Apache-2.0) with a 3-way
classification head (SUPPORTED / CONTRADICTED / NOT_ENOUGH_INFO) and LoRA adapters, in 4-bit
(QLoRA) so it fits a free Colab/Kaggle T4 (16 GB).

1. In the repo run `uv run python scripts/make_verifier_data.py` (needs the teacher labels).
2. Upload `data/finetune/train.jsonl` and `val.jsonl` here (left sidebar, Files) or mount Drive.
3. Runtime > Change runtime type > T4 GPU, then Run all (~30-60 min for 600 steps).
4. Download `verifier-lora.zip` and unzip it into `models/verifier-lora/` in the repo; then
   `uv run python scripts/eval_verifier.py` and set `VERIFIER_BACKEND=small_lora` or `router`.

The format and label order match `bioagent.finetune.model` (`format_input`, `LABELS`).

In [ ]:
!pip -q install "transformers>=5.0" "peft>=0.17" "accelerate>=1.0" "bitsandbytes>=0.45"

In [ ]:
import json
import random
import time
from pathlib import Path

import torch
from peft import LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training
from transformers import AutoModelForSequenceClassification, AutoTokenizer, BitsAndBytesConfig

BASE = "Qwen/Qwen3-1.7B"
LABELS = ["SUPPORTED", "CONTRADICTED", "NOT_ENOUGH_INFO"]  # same order as bioagent.finetune
LABEL2ID = {lab: i for i, lab in enumerate(LABELS)}
STEPS, BATCH, ACCUM, LR, MAX_LEN = 600, 8, 2, 2e-4, 512


# same as bioagent.finetune.model.format_input
def format_input(claim, passage):
    return f"Claim: {claim}\nPassage: {passage}\nDoes the passage support the claim?"


def load(path):
    rows = [json.loads(line) for line in Path(path).read_text().splitlines() if line.strip()]
    return [r for r in rows if (r.get("labels") or {}).get("teacher") in LABEL2ID]


train, val = load("train.jsonl"), load("val.jsonl")
print(len(train), "train /", len(val), "val")

In [ ]:
tok = AutoTokenizer.from_pretrained(BASE)
tok.pad_token = tok.pad_token or tok.eos_token
bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)
model = AutoModelForSequenceClassification.from_pretrained(
    BASE,
    num_labels=3,
    quantization_config=bnb,
    device_map={"": 0},
    id2label=dict(enumerate(LABELS)),
    label2id=LABEL2ID,
)
model.config.pad_token_id = tok.pad_token_id
model = prepare_model_for_kbit_training(model)
model = get_peft_model(
    model,
    LoraConfig(
        task_type=TaskType.SEQ_CLS,
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        target_modules=[
            "q_proj",
            "k_proj",
            "v_proj",
            "o_proj",
            "gate_proj",
            "up_proj",
            "down_proj",
        ],
        modules_to_save=["score"],
    ),
)
model.print_trainable_parameters()

In [ ]:
counts = [sum(r["labels"]["teacher"] == lab for r in train) for lab in LABELS]
weights = torch.tensor([len(train) / (3 * max(1, c)) for c in counts], device="cuda")
loss_fn = torch.nn.CrossEntropyLoss(weight=weights)
opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=LR)
sched = torch.optim.lr_scheduler.LambdaLR(
    opt, lambda s: min(1, (s + 1) / 20) * max(0.05, 1 - s / STEPS)
)


def batch(rows):
    enc = tok(
        [format_input(r["claim"], r["passage"]) for r in rows],
        truncation=True,
        max_length=MAX_LEN,
        padding=True,
        return_tensors="pt",
    ).to("cuda")
    return enc, torch.tensor([LABEL2ID[r["labels"]["teacher"]] for r in rows], device="cuda")


def evaluate():
    model.eval()
    correct = 0
    with torch.inference_mode():
        for i in range(0, len(val), 16):
            enc, y = batch(val[i : i + 16])
            correct += (model(**enc).logits.argmax(-1) == y).sum().item()
    model.train()
    return correct / max(1, len(val))


rng, t0 = random.Random(0), time.time()
model.train()
for step in range(1, STEPS + 1):
    opt.zero_grad()
    for _ in range(ACCUM):
        enc, y = batch(rng.sample(train, BATCH))
        (loss_fn(model(**enc).logits.float(), y) / ACCUM).backward()
    torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
    opt.step()
    sched.step()
    if step % 50 == 0:
        print(
            step,
            f"{(time.time() - t0) / step:.2f}s/step",
            "val acc",
            round(evaluate(), 3),
            f"{torch.cuda.max_memory_allocated() / 1e9:.1f} GB",
        )

In [ ]:
out = Path("verifier-lora")
out.mkdir(exist_ok=True)
model.save_pretrained(out)
tok.save_pretrained(out)
(out / "training.json").write_text(
    json.dumps(
        {
            "base_model": BASE,
            "label_source": "teacher",
            "steps": STEPS,
            "batch": BATCH,
            "accum": ACCUM,
            "lr": LR,
            "max_length": MAX_LEN,
            "quantization": "4-bit nf4 (QLoRA)",
            "device": "cuda (T4)",
            "train_seconds": round(time.time() - t0, 1),
            "val_accuracy": evaluate(),
            "peak_memory_gb": round(torch.cuda.max_memory_allocated() / 1e9, 2),
        },
        indent=1,
    )
)
!zip -qr verifier-lora.zip verifier-lora && ls -lh verifier-lora.zip

Note: an adapter trained on a 4-bit base works best when served on the same 4-bit base; the
repo's `SmallVerifier` loads the bf16 base, which is usually fine for LoRA classifiers but may
shift scores slightly. Re-run `scripts/eval_verifier.py` after copying the adapter.